# PLSQL-Fix: Gemma 4 model server on Kaggle GPU

Runs Ollama with a larger Gemma 4 model on Kaggle's GPUs and opens a temporary HTTPS tunnel,
so the PLSQL-Fix agent on a local machine (where Oracle runs) can call the model.

**Settings before running:** Accelerator = **GPU T4 x2**, Internet = **On**.
Run the cells in order. Cell 5 prints the tunnel URL; cell 6 keeps the session alive.
The tunnel URL is random and stops working when this session ends.

In [ ]:
MODEL = 'gemma4:26b'   # Gemma 4 26B MoE (3.8B active). Alternative: 'gemma4:31b' (dense, slower)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Install Ollama
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!ollama --version

In [ ]:
# 2. Start the Ollama server in the background (both GPUs, one request at a time)
import os, subprocess, time
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_KEEP_ALIVE='60m',
           OLLAMA_NUM_PARALLEL='1', OLLAMA_SCHED_SPREAD='1')
server = subprocess.Popen(['ollama', 'serve'], env=env,
                          stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
!curl -s localhost:11434/api/version

In [ ]:
# 3. Download the model (about 17-20 GB, a few minutes)
!ollama pull {MODEL} 2>&1 | tail -n 2

In [ ]:
# 4. Load it on the GPUs and time one short answer
import requests
t = time.time()
r = requests.post('http://localhost:11434/api/chat', json={
    'model': MODEL, 'stream': False, 'think': False, 'keep_alive': '60m',
    'options': {'num_ctx': 8192, 'temperature': 0.2, 'seed': 1},
    'messages': [{'role': 'user', 'content': 'Write a PL/SQL function that returns the square of a number.'}]},
    timeout=900).json()
print(f"{time.time() - t:.1f} s, {r.get('eval_count')} tokens, "
      f"{r.get('eval_count', 0) / max(r.get('eval_duration', 1) / 1e9, 1e-9):.1f} tokens/s")
print(r['message']['content'][:400])
!ollama ps

In [ ]:
# 5. Open the tunnel and print the URL to use on the laptop
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:11434',
                           '--http-host-header', 'localhost:11434'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        url = m.group(0)
        break
print('TUNNEL URL:', url)
print('On the laptop:  $env:OLLAMA_URL = "' + str(url) + '"')

In [ ]:
# 6. Keep the session alive and show what the server is doing (stop this cell to end)
for i in range(11 * 60):
    time.sleep(60)
    calls = sum(1 for line in open('ollama.log') if '/api/chat' in line)
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip().replace('\n', ' | ')
    print(f"{time.strftime('%H:%M')}  chat calls so far: {calls}  GPU: {gpu}", flush=True)